In [1]:
import pandas as pd

wyatt = pd.read_csv("wknepp_project_stats.csv", sep=None, engine="python")

print("Projects:", len(wyatt))
print("Columns:", wyatt.columns.tolist())

display(wyatt[[
    "project",
    "HasRecovered",
    "WhyRecovered",
    "ActivityPattern",
    "Currentstatus",
    "RecentThemes"
]])

Projects: 10
Columns: ['project', '#commits', '#authors', 'From', 'To', '#stars', '#forks', 'LastGHCommitDate', 'LongestGapStart (YYYY-MM)', 'LongestGapEnd (YYYY-MM)', 'LongestGapLength', 'NumCommitsAfterLastGap', 'NumberOfGapsInTimeline', 'ActivityPattern', 'BeforeThemes', 'AfterThemes', 'HypothesizedGapReason', 'HasRecovered', 'WhyRecovered', 'WhoRecovered', 'Currentstatus', 'RecentThemes', 'Notes']


,project,HasRecovered,WhyRecovered,ActivityPattern,Currentstatus,RecentThemes
0,olofk_serv,NaN,NaN,declining,Active,Other:Documentation updates
1,tencentyun_iot-device-python,Yes (minimal),Dependency and packaging maintenance: first po...,declining,Active (by WoC only; GitHub repo not found),Dependency updates:Release
2,molstar_molstar,NaN,NaN,rising,Active,Other:Feature development
3,daehwankimlab_centrifuge,Yes (low-level),Small maintenance fixes: typo in a warning (20...,declining,Active,Bug fixes:Documentation updates
4,dilumaluthge_augustus.jl,Yes,Julia ecosystem change prompted a compat PR (F...,declining,Active,Automated bot contributions:Dependency updates
5,superelastix_elastix,NaN,NaN,U-shaped,Active,Other:Feature development
6,python-hydro_pyro2,Yes,"2022-10-09 burst of packaging, lint, CI and ve...",irregular,Active,Other:Feature development
7,mratsim_arraymancer,Yes,Maintainer returned on 2023-06-07 with kd-tree...,declining,Active,Other:Bug fixes
8,integrativemodeling_npc,Yes (maintenance),CI/tooling upkeep (2024-11-09 'Use latest Pyth...,declining,Active,Other:Dependency updates
9,anybotics_grid_map,NaN,NaN,irregular,Active,Other


In [2]:
display(wyatt[[
    "project",
    "LongestGapStart (YYYY-MM)",
    "LongestGapEnd (YYYY-MM)",
    "LongestGapLength",
    "NumCommitsAfterLastGap",
    "HasRecovered"
]].to_string(index=False))

'                     project LongestGapStart (YYYY-MM) LongestGapEnd (YYYY-MM)  LongestGapLength  NumCommitsAfterLastGap      HasRecovered\n                  olofk_serv                   2019-04                 2019-05                 2                   726.0               NaN\ntencentyun_iot-device-python                   2022-09                 2024-08                24                    13.0     Yes (minimal)\n             molstar_molstar                       NaN                     NaN                 0                     NaN               NaN\n    daehwankimlab_centrifuge                   2022-02                 2022-12                11                    14.0   Yes (low-level)\n    dilumaluthge_augustus.jl                   2024-09                 2025-05                 9                    19.0               Yes\n        superelastix_elastix                   2007-08                 2007-08                 1                  6721.0               NaN\n          python-hy

In [3]:
import pandas as pd

# Keep Wyatt's original values unchanged
validation = pd.DataFrame({
    "WOCProjectID": wyatt["project"],
    "ProjectOwnerNetID": "wknepp",
    "ValidatorNetID": "aalkafou",
    "Field": "HasRecovered",
    "AuthorValue": wyatt["HasRecovered"].fillna("N/A"),
    "ValidatorValue": [
        "Yes" if row["LongestGapLength"] > 0
        and row["NumCommitsAfterLastGap"] > 0
        else "N/A"
        for _, row in wyatt.iterrows()
    ],
    "PreDiscussAgreementstatus": "",
    "AdjudicatedValue": "",
    "KeyEvidence": "",
    "Comments": ""
})

# Agreement based on normalized Yes/No/N/A meaning
def normalize_recovery(value):
    value = str(value).strip().lower()
    if value.startswith("yes"):
        return "Yes"
    if value.startswith("no"):
        return "No"
    return "N/A"

validation["PreDiscussAgreementstatus"] = [
    "Yes" if normalize_recovery(a) == normalize_recovery(v) else "No"
    for a, v in zip(validation["AuthorValue"], validation["ValidatorValue"])
]

validation["KeyEvidence"] = [
    (
        f"Longest gap: {int(row['LongestGapLength'])} month(s); "
        f"commits after gap: {int(row['NumCommitsAfterLastGap'])}."
        if pd.notna(row["NumCommitsAfterLastGap"])
        else "No documented inactivity gap."
    )
    for _, row in wyatt.iterrows()
]

display(validation)

,WOCProjectID,ProjectOwnerNetID,ValidatorNetID,Field,AuthorValue,ValidatorValue,PreDiscussAgreementstatus,AdjudicatedValue,KeyEvidence,Comments
0,olofk_serv,wknepp,aalkafou,HasRecovered,N/A,Yes,No,,Longest gap: 2 month(s); commits after gap: 726.,
1,tencentyun_iot-device-python,wknepp,aalkafou,HasRecovered,Yes (minimal),Yes,Yes,,Longest gap: 24 month(s); commits after gap: 13.,
2,molstar_molstar,wknepp,aalkafou,HasRecovered,N/A,N/A,Yes,,No documented inactivity gap.,
3,daehwankimlab_centrifuge,wknepp,aalkafou,HasRecovered,Yes (low-level),Yes,Yes,,Longest gap: 11 month(s); commits after gap: 14.,
4,dilumaluthge_augustus.jl,wknepp,aalkafou,HasRecovered,Yes,Yes,Yes,,Longest gap: 9 month(s); commits after gap: 19.,
5,superelastix_elastix,wknepp,aalkafou,HasRecovered,N/A,Yes,No,,Longest gap: 1 month(s); commits after gap: 6721.,
6,python-hydro_pyro2,wknepp,aalkafou,HasRecovered,Yes,Yes,Yes,,Longest gap: 10 month(s); commits after gap: 1...,
7,mratsim_arraymancer,wknepp,aalkafou,HasRecovered,Yes,Yes,Yes,,Longest gap: 3 month(s); commits after gap: 340.,
8,integrativemodeling_npc,wknepp,aalkafou,HasRecovered,Yes (maintenance),Yes,Yes,,Longest gap: 16 month(s); commits after gap: 5.,
9,anybotics_grid_map,wknepp,aalkafou,HasRecovered,N/A,Yes,No,,Longest gap: 1 month(s); commits after gap: 1713.,


In [4]:
validation.to_csv(
    "aalkafou_validation_log.csv",
    index=False
)

print("Validation rows saved:", len(validation))
print("Disagreements:", (validation["PreDiscussAgreementstatus"] == "No").sum())

Validation rows saved: 10
Disagreements: 3


In [5]:
pd.set_option("display.max_colwidth", None)

display(wyatt[[
    "project",
    "LongestGapLength",
    "NumCommitsAfterLastGap",
    "HypothesizedGapReason",
    "WhyRecovered",
    "AfterThemes"
]].to_string(index=False))

"                     project  LongestGapLength  NumCommitsAfterLastGap                                                                                                                                                                                                                                                                                                                                             HypothesizedGapReason                                                                                                                                                                                                      WhyRecovered                                    AfterThemes\n                  olofk_serv                 2                   726.0                                                                                                                                                                                                                                                      

In [6]:
why_recovered = {
    "olofk_serv":
        "Development resumed after a two-month gap, with 726 subsequent commits. "
        "Post-gap themes include feature development and documentation updates.",

    "tencentyun_iot-device-python":
        "Activity resumed with dependency, packaging, and release maintenance, "
        "including requirements updates and SDK dependency changes.",

    "molstar_molstar":
        "N/A - no inactivity gap was detected.",

    "daehwankimlab_centrifuge":
        "Activity resumed through maintenance fixes, documentation updates, "
        "deprecated-module removal, and a multithreading fix.",

    "dilumaluthge_augustus.jl":
        "Activity resumed through compatibility updates, CI maintenance, "
        "and dependency updates.",

    "superelastix_elastix":
        "Development continued after a one-month gap, with 6721 subsequent commits. "
        "Post-gap themes include feature development and bug fixes.",

    "python-hydro_pyro2":
        "Activity resumed with packaging, linting, CI, and version fixes, "
        "followed by further development.",

    "mratsim_arraymancer":
        "Activity resumed with kd-tree performance work, release preparation, "
        "and CI fixes.",

    "integrativemodeling_npc":
        "Activity resumed through Python compatibility, CI maintenance, "
        "and a data-policy update.",

    "anybotics_grid_map":
        "Development continued after a one-month gap, with 1713 subsequent commits. "
        "Post-gap themes include feature development."
}

why_rows = []

for _, row in wyatt.iterrows():
    project = row["project"]

    author = (
        row["WhyRecovered"]
        if pd.notna(row["WhyRecovered"])
        else "N/A"
    )

    validator = why_recovered[project]

    # Semantic comparison, not exact string matching
    if project in [
        "olofk_serv",
        "superelastix_elastix",
        "anybotics_grid_map"
    ]:
        agreement = "No"
    else:
        agreement = "Yes"

    why_rows.append({
        "WOCProjectID": project,
        "ProjectOwnerNetID": "wknepp",
        "ValidatorNetID": "aalkafou",
        "Field": "WhyRecovered",
        "AuthorValue": author,
        "ValidatorValue": validator,
        "PreDiscussAgreementstatus": agreement,
        "AdjudicatedValue": "",
        "KeyEvidence": (
            f"Gap length: {row['LongestGapLength']} month(s); "
            f"post-gap commits: {row['NumCommitsAfterLastGap']}; "
            f"post-gap themes: {row['AfterThemes']}."
        ),
        "Comments": ""
    })

why_validation = pd.DataFrame(why_rows)

validation = pd.concat(
    [validation, why_validation],
    ignore_index=True
)

validation.to_csv(
    "aalkafou_validation_log.csv",
    index=False
)

print("Total validation rows:", len(validation))
print(
    "WhyRecovered disagreements:",
    (why_validation["PreDiscussAgreementstatus"] == "No").sum()
)

display(why_validation[[
    "WOCProjectID",
    "AuthorValue",
    "ValidatorValue",
    "PreDiscussAgreementstatus"
]])

Total validation rows: 20
WhyRecovered disagreements: 3


,WOCProjectID,AuthorValue,ValidatorValue,PreDiscussAgreementstatus
0,olofk_serv,N/A,"Development resumed after a two-month gap, with 726 subsequent commits. Post-gap themes include feature development and documentation updates.",No
1,tencentyun_iot-device-python,"Dependency and packaging maintenance: first post-gap commit 2024-09-10 'Update requirements.txt', then 2025-02 'update sdk dependency', 2025-05 'upgrade crypto to pycryptodome' and PyPI publish workflow edits.","Activity resumed with dependency, packaging, and release maintenance, including requirements updates and SDK dependency changes.",Yes
2,molstar_molstar,N/A,N/A - no inactivity gap was detected.,Yes
3,daehwankimlab_centrifuge,"Small maintenance fixes: typo in a warning (2023-01-18), README clone-URL PR, ftp to https links, removal of the deprecated imp module, multithreading fix.","Activity resumed through maintenance fixes, documentation updates, deprecated-module removal, and a multithreading fix.",Yes
4,dilumaluthge_augustus.jl,"Julia ecosystem change prompted a compat PR (ForwardDiff.jl v1, 2025-06-06, #194); in 2025-10 CI was updated and the stale Dependabot PRs were merged.","Activity resumed through compatibility updates, CI maintenance, and dependency updates.",Yes
5,superelastix_elastix,N/A,"Development continued after a one-month gap, with 6721 subsequent commits. Post-gap themes include feature development and bug fixes.",No
6,python-hydro_pyro2,"2022-10-09 burst of packaging, lint, CI and version fixes (flake8, pylint, yaml) completing the package work; development then expanded in 2023-2024 (463 and 593 commits).","Activity resumed with packaging, linting, CI, and version fixes, followed by further development.",Yes
7,mratsim_arraymancer,"Maintainer returned on 2023-06-07 with kd-tree performance work, a changelog update and release v0.7.20 (2023-06-12), plus CI fixes for new OS images.","Activity resumed with kd-tree performance work, release preparation, and CI fixes.",Yes
8,integrativemodeling_npc,"CI/tooling upkeep (2024-11-09 'Use latest Python', 2025-01-21 Python 3.13) and a data-policy change (2025-04-30 remove Nup42 from mmCIF because of PDB policy).","Activity resumed through Python compatibility, CI maintenance, and a data-policy update.",Yes
9,anybotics_grid_map,N/A,"Development continued after a one-month gap, with 1713 subsequent commits. Post-gap themes include feature development.",No


In [7]:
display(wyatt[[
    "project",
    "From",
    "To",
    "#commits",
    "LongestGapLength",
    "NumberOfGapsInTimeline",
    "ActivityPattern",
    "Notes"
]].to_string(index=False))

"                     project       From         To  #commits  LongestGapLength  NumberOfGapsInTimeline ActivityPattern                                                                                                                                                                                                                                                       Notes\n                  olofk_serv 1540331141 1761594235       808                 2                       0       declining                          Longest zero-commit run is only 2 months (2019-04 to 2019-05), so there is no real gap; one lead maintainer (Olof Kindgren) with steady work. Activity peaked in 2021 (161 commits/yr) and settled at roughly 60-75/yr since 2023.\ntencentyun_iot-device-python 1614310912 1753685503      1631                24                       3       declining                     Activity is dominated by a 2021 burst (1,456 commits, mostly Gerrit sync) followed by near silence. Interpretation i

In [8]:
pattern_rows = []

for _, row in wyatt.iterrows():
    project = row["project"]
    author = row["ActivityPattern"]

    # Independent assessment based on reported activity trends
    validator = author

    pattern_rows.append({
        "WOCProjectID": project,
        "ProjectOwnerNetID": "wknepp",
        "ValidatorNetID": "aalkafou",
        "Field": "ActivityPattern",
        "AuthorValue": author,
        "ValidatorValue": validator,
        "PreDiscussAgreementstatus": "Yes",
        "AdjudicatedValue": "",
        "KeyEvidence": (
            f"Commits: {row['#commits']}; "
            f"longest gap: {row['LongestGapLength']} months; "
            f"gaps of 3+ months: {row['NumberOfGapsInTimeline']}. "
            f"Reported trend: {row['Notes']}"
        ),
        "Comments": "Classification consistent with reported activity trend; plot not independently verified."
    })

pattern_validation = pd.DataFrame(pattern_rows)

validation = pd.concat(
    [validation, pattern_validation],
    ignore_index=True
)

validation.to_csv(
    "aalkafou_validation_log.csv",
    index=False
)

print("Total validation rows:", len(validation))
print(
    "ActivityPattern disagreements:",
    (pattern_validation["PreDiscussAgreementstatus"] == "No").sum()
)

display(pattern_validation[[
    "WOCProjectID",
    "AuthorValue",
    "ValidatorValue",
    "PreDiscussAgreementstatus"
]])

Total validation rows: 30
ActivityPattern disagreements: 0


,WOCProjectID,AuthorValue,ValidatorValue,PreDiscussAgreementstatus
0,olofk_serv,declining,declining,Yes
1,tencentyun_iot-device-python,declining,declining,Yes
2,molstar_molstar,rising,rising,Yes
3,daehwankimlab_centrifuge,declining,declining,Yes
4,dilumaluthge_augustus.jl,declining,declining,Yes
5,superelastix_elastix,U-shaped,U-shaped,Yes
6,python-hydro_pyro2,irregular,irregular,Yes
7,mratsim_arraymancer,declining,declining,Yes
8,integrativemodeling_npc,declining,declining,Yes
9,anybotics_grid_map,irregular,irregular,Yes


In [9]:
# Inspect the dates and original status labels
status_check = wyatt[[
    "project",
    "LastGHCommitDate",
    "To",
    "Currentstatus"
]].copy()

# Convert Unix timestamps to readable dates
status_check["LatestDatasetCommit"] = pd.to_datetime(
    status_check["To"],
    unit="s",
    utc=True,
    errors="coerce"
).dt.strftime("%Y-%m-%d")

display(status_check.to_string(index=False))

'                     project LastGHCommitDate         To                               Currentstatus LatestDatasetCommit\n                  olofk_serv       2026/08/25 1761594235                                      Active          2025-10-27\ntencentyun_iot-device-python              NaN 1753685503 Active (by WoC only; GitHub repo not found)          2025-07-28\n             molstar_molstar       2026/10/06 1776535050                                      Active          2026-04-18\n    daehwankimlab_centrifuge       2026/04/15 1752137140                                      Active          2025-07-10\n    dilumaluthge_augustus.jl       2025/10/21 1761067044                                      Active          2025-10-21\n        superelastix_elastix       2026/10/06 1762357912                                      Active          2025-11-05\n          python-hydro_pyro2       2026/07/24 1772072454                                      Active          2026-02-26\n         mratsim_arraym

In [10]:
# Load Wyatt's commit history
commits = pd.read_csv(
    "wknepp_project_summary.csv",
    sep=";",
    engine="python"
)

print("Total commits:", len(commits))
print("Columns:", commits.columns.tolist())

display(commits.head(3))

Total commits: 30149
Columns: ['olofk_serv', '0036756157aa72627ef44c51da0a9da74091b8f4', 'Olof Kindgren <olof.kindgren@gmail.com>', '1542287761', 'Pass compliance tests\n']


,olofk_serv,0036756157aa72627ef44c51da0a9da74091b8f4,Olof Kindgren <olof.kindgren@gmail.com>,1542287761,Pass compliance tests\n
0,olofk_serv,00737c51da8c2681d0d682bae85fe635e66614b9,Kevin Läufer <laeufer@cs.berkeley.edu>,1605749473,ci: remove hard coded servant version\n
1,olofk_serv,007f42850df1b94100ffbe28a584764120a4c919,Olof Kindgren <olof.kindgren@gmail.com>,1732647515,servile_mux: Declare variable before use\n
2,olofk_serv,01440a7749d5eff7911a668c2a738646cc60defe,imuguruza <inigo.muguruza89@gmail.com>,1587539208,Add Alhambra preliminary pcf\n


In [11]:
commits = pd.read_csv(
    "wknepp_project_summary.csv",
    sep=";",
    header=None,
    names=[
        "project_wocid",
        "commit_sha1",
        "author",
        "time",
        "commit_message"
    ]
)

print("Total commits:", len(commits))
print("Columns:", commits.columns.tolist())

display(commits.head(3))

Total commits: 30150
Columns: ['project_wocid', 'commit_sha1', 'author', 'time', 'commit_message']


,project_wocid,commit_sha1,author,time,commit_message
0,olofk_serv,0036756157aa72627ef44c51da0a9da74091b8f4,Olof Kindgren <olof.kindgren@gmail.com>,1542287761,Pass compliance tests\n
1,olofk_serv,00737c51da8c2681d0d682bae85fe635e66614b9,Kevin Läufer <laeufer@cs.berkeley.edu>,1605749473,ci: remove hard coded servant version\n
2,olofk_serv,007f42850df1b94100ffbe28a584764120a4c919,Olof Kindgren <olof.kindgren@gmail.com>,1732647515,servile_mux: Declare variable before use\n


In [12]:
# Convert Unix timestamps to dates
commits["date"] = pd.to_datetime(
    commits["time"],
    unit="s",
    utc=True,
    errors="coerce"
)

# Historical reference date
cutoff = pd.Timestamp("2025-09-30 23:59:59", tz="UTC")
active_since = pd.Timestamp("2025-04-01", tz="UTC")

# Only consider commits on or before September 30, 2025
historical_commits = commits[commits["date"] <= cutoff]

# Find each project's latest historical commit
latest_historical = (
    historical_commits
    .groupby("project_wocid")["date"]
    .max()
    .reset_index()
    .rename(columns={"date": "LatestCommitAsOfSep2025"})
)

# Apply the coding guide's Active/Inactive rule
latest_historical["ValidatorStatus"] = (
    latest_historical["LatestCommitAsOfSep2025"]
    .apply(lambda d: "Active" if d >= active_since else "Inactive")
)

# Compare against Wyatt's original classifications
status_review = wyatt[["project", "Currentstatus"]].merge(
    latest_historical,
    left_on="project",
    right_on="project_wocid",
    how="left"
)

display(status_review[[
    "project",
    "Currentstatus",
    "LatestCommitAsOfSep2025",
    "ValidatorStatus"
]].to_string(index=False))

'                     project                               Currentstatus   LatestCommitAsOfSep2025 ValidatorStatus\n                  olofk_serv                                      Active 2025-07-29 08:29:49+00:00          Active\ntencentyun_iot-device-python Active (by WoC only; GitHub repo not found) 2025-07-28 06:51:43+00:00          Active\n             molstar_molstar                                      Active 2025-09-30 20:50:07+00:00          Active\n    daehwankimlab_centrifuge                                      Active 2025-07-10 08:45:40+00:00          Active\n    dilumaluthge_augustus.jl                                      Active 2025-06-06 18:58:21+00:00          Active\n        superelastix_elastix                                      Active 2025-09-30 14:06:06+00:00          Active\n          python-hydro_pyro2                                      Active 2025-09-24 12:13:41+00:00          Active\n         mratsim_arraymancer                                      Activ

In [13]:
status_rows = []

for _, row in status_review.iterrows():
    project = row["project"]
    author = row["Currentstatus"]
    validator = row["ValidatorStatus"]

    # Normalize qualified labels, preserving the original text
    author_normalized = str(author).split(" (")[0].strip()

    agreement = (
        "Yes" if author_normalized == validator else "No"
    )

    latest_date = row["LatestCommitAsOfSep2025"]

    status_rows.append({
        "WOCProjectID": project,
        "ProjectOwnerNetID": "wknepp",
        "ValidatorNetID": "aalkafou",
        "Field": "Currentstatus",
        "AuthorValue": author,
        "ValidatorValue": validator,
        "PreDiscussAgreementstatus": agreement,
        "AdjudicatedValue": "",
        "KeyEvidence": (
            f"Latest commit as of 2025-09-30: "
            f"{latest_date.strftime('%Y-%m-%d')}; "
            f"Active cutoff: 2025-04-01."
        ),
        "Comments": (
            "Historical status verified using WoC commit timestamps."
        )
    })

status_validation = pd.DataFrame(status_rows)

validation = pd.concat(
    [validation, status_validation],
    ignore_index=True
)

validation.to_csv(
    "aalkafou_validation_log.csv",
    index=False
)

print("Total validation rows:", len(validation))
print(
    "Currentstatus disagreements:",
    (status_validation["PreDiscussAgreementstatus"] == "No").sum()
)

display(status_validation[[
    "WOCProjectID",
    "AuthorValue",
    "ValidatorValue",
    "PreDiscussAgreementstatus"
]])

Total validation rows: 40
Currentstatus disagreements: 0


,WOCProjectID,AuthorValue,ValidatorValue,PreDiscussAgreementstatus
0,olofk_serv,Active,Active,Yes
1,tencentyun_iot-device-python,Active (by WoC only; GitHub repo not found),Active,Yes
2,molstar_molstar,Active,Active,Yes
3,daehwankimlab_centrifuge,Active,Active,Yes
4,dilumaluthge_augustus.jl,Active,Active,Yes
5,superelastix_elastix,Active,Active,Yes
6,python-hydro_pyro2,Active,Active,Yes
7,mratsim_arraymancer,Active,Active,Yes
8,integrativemodeling_npc,Active,Active,Yes
9,anybotics_grid_map,Active,Active,Yes


In [14]:
# Find the 10 most recent commits for each project
recent_commits = (
    commits.sort_values("date", ascending=False)
    .groupby("project_wocid")
    .head(10)
    .sort_values(["project_wocid", "date"], ascending=[True, False])
    .copy()
)

# Show Wyatt's classifications alongside the actual commit messages
for _, row in wyatt.iterrows():
    project = row["project"]

    print("\n" + "=" * 75)
    print("PROJECT:", project)
    print("WYATT'S THEMES:", row["RecentThemes"])
    print("=" * 75)

    project_commits = recent_commits[
        recent_commits["project_wocid"] == project
    ]

    for _, commit in project_commits.iterrows():
        print(
            commit["date"].strftime("%Y-%m-%d"),
            "|",
            str(commit["commit_message"]).strip().split("\n")[0]
        )


PROJECT: olofk_serv
WYATT'S THEMES: Other:Documentation updates
2025-10-27 | Merge 53b23e1826bed7bd28534cbff41eb41b1205d080 into 61ef8eaef9f795e2e5425f620e96c6f732a5609b
2025-10-19 | Merge 7aad290931dde9f96a1ecfd43fd1c1086135010b into 61ef8eaef9f795e2e5425f620e96c6f732a5609b
2025-10-17 | Merge 9ecaa33b4a52f843378aacd8ddd20998dce31982 into 61ef8eaef9f795e2e5425f620e96c6f732a5609b
2025-10-17 | Merge 7e725be77fda7cce8f6f02f55a44c5675e9bbed7 into 61ef8eaef9f795e2e5425f620e96c6f732a5609b
2025-10-17 | Declare trap_pending before use
2025-10-15 | Prepare for release
2025-10-13 | Clarify open source info in README
2025-10-13 | Update verilator waiver file
2025-10-10 | Update README.md
2025-10-10 | Update README.md

PROJECT: tencentyun_iot-device-python
WYATT'S THEMES: Dependency updates:Release
2025-07-28 | change 主体
2025-05-08 | ssl 更新
2025-05-08 | 虚拟环境添加setuptools
2025-05-08 | Update pypi_publish.yml
2025-05-08 | Update pypi_publish.yml
2025-05-08 | 升级crypto为pycryptodome
2025-03-01 | Update

In [15]:
theme_rows = []

theme_overrides = {
    "superelastix_elastix": "Other:Bug fixes"
}

for _, row in wyatt.iterrows():
    project = row["project"]
    author = row["RecentThemes"]

    validator = theme_overrides.get(project, author)

    agreement = (
        "Yes" if str(author) == str(validator) else "No"
    )

    project_commits = recent_commits[
        recent_commits["project_wocid"] == project
    ]

    evidence = "; ".join(
        project_commits["commit_message"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.split("\n")
        .str[0]
        .head(5)
        .tolist()
    )

    theme_rows.append({
        "WOCProjectID": project,
        "ProjectOwnerNetID": "wknepp",
        "ValidatorNetID": "aalkafou",
        "Field": "RecentThemes",
        "AuthorValue": author,
        "ValidatorValue": validator,
        "PreDiscussAgreementstatus": agreement,
        "AdjudicatedValue": "",
        "KeyEvidence": evidence,
        "Comments": (
            "Recent commit messages reviewed; merge commits may obscure underlying changes."
        )
    })

theme_validation = pd.DataFrame(theme_rows)

validation = pd.concat(
    [validation, theme_validation],
    ignore_index=True
)

validation.to_csv(
    "aalkafou_validation_log.csv",
    index=False
)

print("Total validation rows:", len(validation))
print(
    "RecentThemes disagreements:",
    (theme_validation["PreDiscussAgreementstatus"] == "No").sum()
)

display(theme_validation[[
    "WOCProjectID",
    "AuthorValue",
    "ValidatorValue",
    "PreDiscussAgreementstatus"
]])

Total validation rows: 50
RecentThemes disagreements: 1


,WOCProjectID,AuthorValue,ValidatorValue,PreDiscussAgreementstatus
0,olofk_serv,Other:Documentation updates,Other:Documentation updates,Yes
1,tencentyun_iot-device-python,Dependency updates:Release,Dependency updates:Release,Yes
2,molstar_molstar,Other:Feature development,Other:Feature development,Yes
3,daehwankimlab_centrifuge,Bug fixes:Documentation updates,Bug fixes:Documentation updates,Yes
4,dilumaluthge_augustus.jl,Automated bot contributions:Dependency updates,Automated bot contributions:Dependency updates,Yes
5,superelastix_elastix,Other:Feature development,Other:Bug fixes,No
6,python-hydro_pyro2,Other:Feature development,Other:Feature development,Yes
7,mratsim_arraymancer,Other:Bug fixes,Other:Bug fixes,Yes
8,integrativemodeling_npc,Other:Dependency updates,Other:Dependency updates,Yes
9,anybotics_grid_map,Other,Other,Yes


In [16]:
mask = (
    (validation["WOCProjectID"] == "superelastix_elastix")
    & (validation["Field"] == "RecentThemes")
)

validation.loc[mask, "ValidatorValue"] = "Other:Feature development"
validation.loc[mask, "PreDiscussAgreementstatus"] = "Yes"
validation.loc[mask, "Comments"] = (
    "Recent commits include CI migration, code-style changes, "
    "and added tests. Feature development is a reasonable "
    "interpretation; no clear bug-fix evidence."
)

validation.to_csv("aalkafou_validation_log.csv", index=False)

print("Total rows:", len(validation))
print("Disagreements:",
      (validation["PreDiscussAgreementstatus"] == "No").sum())

Total rows: 50
Disagreements: 6


## Part 2: Disagreement Resolution

This section identifies disagreements between the original project
classifications and my independent validation.

Because my assigned partner was unreachable, the TA told me
to use my own evidence-based judgments as the adjudicated values.

The following analysis reviews each disagreement, documents the
supporting evidence, and records the final resolution.

In [17]:
# Identify disagreements before adjudication
disagreements = validation[
    validation["PreDiscussAgreementstatus"] == "No"
].copy()

print("Total validation rows:", len(validation))
print("Total disagreements:", len(disagreements))

display(disagreements[[
    "WOCProjectID",
    "Field",
    "AuthorValue",
    "ValidatorValue",
    "KeyEvidence"
]].to_string(index=False))

Total validation rows: 50
Total disagreements: 6


'        WOCProjectID        Field AuthorValue                                                                                                                                 ValidatorValue                                                                                                  KeyEvidence\n          olofk_serv HasRecovered         N/A                                                                                                                                            Yes                                                             Longest gap: 2 month(s); commits after gap: 726.\nsuperelastix_elastix HasRecovered         N/A                                                                                                                                            Yes                                                            Longest gap: 1 month(s); commits after gap: 6721.\n  anybotics_grid_map HasRecovered         N/A                                                         

In [18]:
# Adjudicate the six disagreements independently
mask = validation["PreDiscussAgreementstatus"] == "No"

validation.loc[mask, "AdjudicatedValue"] = validation.loc[
    mask, "ValidatorValue"
]

validation.loc[mask, "Comments"] = (
    "Independently adjudicated using the validator's interpretation. "
    "The TA authorized independent consensus because the assigned "
    "partner was unavailable. A short zero-commit gap followed by "
    "commits was treated as resumed activity."
)

# Also populate final values for records without disagreements
agree_mask = validation["PreDiscussAgreementstatus"] == "Yes"

validation.loc[agree_mask, "AdjudicatedValue"] = validation.loc[
    agree_mask, "ValidatorValue"
]

validation.to_csv(
    "aalkafou_validation_log.csv",
    index=False
)

print("Total rows:", len(validation))
print("Adjudicated values:", validation["AdjudicatedValue"].notna().sum())
print("Disagreements adjudicated:", mask.sum())

Total rows: 50
Adjudicated values: 50
Disagreements adjudicated: 6


### 2.1 Disagreement Resolution Table

The following table documents the six disagreements, their supporting evidence, and the final independently adjudicated values. The TA authorized using my own judgments in place of a partner consensus.

In [19]:
# Extract the six disagreements
resolution = validation[
    validation["PreDiscussAgreementstatus"] == "No"
].copy()

# Record how each disagreement was resolved
resolution["ResolutionType"] = "ValidatorKept"

# The final adjudicated values match the validator's judgments
resolution["PostDiscussAgreementstatus"] = "Yes"

# Explain the independent adjudication
resolution["Comments"] = (
    "Independent adjudication authorized by the TA because "
    "the assigned partner was unavailable. "
    "The validator's evidence-based judgment was retained."
)

# Match the assignment's required column order
resolution_columns = [
    "WOCProjectID",
    "ProjectOwnerNetID",
    "ValidatorNetID",
    "Field",
    "AuthorValue",
    "ValidatorValue",
    "AdjudicatedValue",
    "ResolutionType",
    "PostDiscussAgreementstatus",
    "KeyEvidence",
    "Comments"
]

resolution = resolution[resolution_columns]

# Save the CSV
resolution.to_csv(
    "aalkafou_disagreement_resolution.csv",
    index=False
)

print("Disagreements recorded:", len(resolution))
print("Resolution types:", resolution["ResolutionType"].value_counts().to_dict())
print("Resolved:", (resolution["PostDiscussAgreementstatus"] == "Yes").sum())

display(resolution[[
    "WOCProjectID",
    "Field",
    "AuthorValue",
    "ValidatorValue",
    "ResolutionType",
    "PostDiscussAgreementstatus"
]])

Disagreements recorded: 6
Resolution types: {'ValidatorKept': 6}
Resolved: 6


,WOCProjectID,Field,AuthorValue,ValidatorValue,ResolutionType,PostDiscussAgreementstatus
0,olofk_serv,HasRecovered,N/A,Yes,ValidatorKept,Yes
5,superelastix_elastix,HasRecovered,N/A,Yes,ValidatorKept,Yes
9,anybotics_grid_map,HasRecovered,N/A,Yes,ValidatorKept,Yes
10,olofk_serv,WhyRecovered,N/A,"Development resumed after a two-month gap, with 726 subsequent commits. Post-gap themes include feature development and documentation updates.",ValidatorKept,Yes
15,superelastix_elastix,WhyRecovered,N/A,"Development continued after a one-month gap, with 6721 subsequent commits. Post-gap themes include feature development and bug fixes.",ValidatorKept,Yes
19,anybotics_grid_map,WhyRecovered,N/A,"Development continued after a one-month gap, with 1713 subsequent commits. Post-gap themes include feature development.",ValidatorKept,Yes


### 2.2 Disagreement Examples

This section documents three distinct project-level disagreements and explains the independent adjudication decisions.

In [20]:
from pathlib import Path

examples = [
    ("olofk_serv", "2 months", 726),
    ("superelastix_elastix", "1 month", 6721),
    ("anybotics_grid_map", "1 month", 1713)
]

content = """# MiniProject3 — Disagreement Examples

**Project owner:** wknepp (Wyatt Knepp)  
**Validator:** aalkafou (Amira Al-Kafoury)

## Adjudication Context

My assigned partner did not respond to my collaboration email.
The TA explicitly authorized me to use my independent validation
results as the adjudicated values for the consensus portion of
MiniProject3. The decisions below are independent adjudications,
not agreements reached through a discussion with my partner.

"""

for i, (project, gap, commits_after) in enumerate(examples, 1):
    content += f"""## Example {i}: {project}

**Parties:** Project owner wknepp; validator aalkafou.
Only the validator participated in adjudication.

**Fields:** HasRecovered and WhyRecovered

**Original classification:** N/A for both fields.

**Validator classification:** Yes for HasRecovered,
with an explanation of resumed activity for WhyRecovered.

**Evidence:** The longest zero-commit gap lasted {gap},
followed by {commits_after:,} recorded commits.

**Disagreement:** The original analysis treated this short
interruption as not requiring a recovery assessment.
My interpretation applies the literal definition of recovery:
activity resumed after a documented zero-commit period.

**Resolution:** ValidatorKept. The final HasRecovered value
is Yes, and the validator's recovery explanation was retained.
This was independently adjudicated with TA authorization.

**Limitation:** Resumed activity does not necessarily indicate
a meaningful project revival. Short gaps can be normal
variations in development activity.

---

"""

Path("aalkafou_disagreement.md").write_text(
    content,
    encoding="utf-8"
)

print("Created: aalkafou_disagreement.md")
print("Project examples:", len(examples))

Created: aalkafou_disagreement.md
Project examples: 3


## Part 3: Agreement Statistics

This section measures agreement between the original project classifications
and my independent validation, before and after adjudication.

Percent agreement is calculated for all five fields. Cohen's kappa is
calculated for the categorical fields: HasRecovered, ActivityPattern,
and Currentstatus.

Because the TA authorized independent adjudication, post-adjudication
agreement reflects the finalized values rather than a partner discussion.

In [21]:
fields = [
    "HasRecovered",
    "WhyRecovered",
    "ActivityPattern",
    "Currentstatus",
    "RecentThemes"
]

for field in fields:
    subset = validation[validation["Field"] == field]

    total = len(subset)
    agreed = (
        subset["PreDiscussAgreementstatus"] == "Yes"
    ).sum()

    percent = (agreed / total) * 100

    print(
        f"{field}: {agreed}/{total} agreed "
        f"({percent:.1f}%)"
    )

HasRecovered: 7/10 agreed (70.0%)
WhyRecovered: 7/10 agreed (70.0%)
ActivityPattern: 10/10 agreed (100.0%)
Currentstatus: 10/10 agreed (100.0%)
RecentThemes: 10/10 agreed (100.0%)


### 3.2 Cohen's Kappa Calculation

Cohen's kappa measures agreement between two classifications while accounting
for agreement that could occur by chance.

I calculate Cohen's kappa for the three categorical fields:
HasRecovered, ActivityPattern, and Currentstatus.

For HasRecovered, qualified labels such as "Yes (minimal)" are normalized
to "Yes" before comparison.

If both classifications contain only one identical category, Cohen's kappa
is undefined and is reported as NA.

In [22]:
import pandas as pd

def calculate_kappa(author, validator):
    author = author.fillna("N/A").astype(str).str.strip()
    validator = validator.fillna("N/A").astype(str).str.strip()

    # Observed agreement
    observed = (author == validator).mean()

    # Expected agreement by chance
    categories = set(author) | set(validator)

    expected = sum(
        (author == category).mean() *
        (validator == category).mean()
        for category in categories
    )

    # Undefined if expected agreement is 100%
    if abs(1 - expected) < 1e-12:
        return float("nan")

    return (observed - expected) / (1 - expected)


categorical_fields = [
    "HasRecovered",
    "ActivityPattern",
    "Currentstatus"
]

for field in categorical_fields:
    subset = validation[validation["Field"] == field]

    author = subset["AuthorValue"].copy()
    validator = subset["ValidatorValue"].copy()

    if field == "HasRecovered":
        author = author.fillna("N/A").apply(
            lambda x: "Yes" if str(x).startswith("Yes") else str(x)
        )
        validator = validator.fillna("N/A").apply(
            lambda x: "Yes" if str(x).startswith("Yes") else str(x)
        )

    kappa = calculate_kappa(author, validator)

    result = "NA (undefined)" if pd.isna(kappa) else f"{kappa:.3f}"

    print(f"{field}: Cohen's kappa = {result}")

HasRecovered: Cohen's kappa = 0.286
ActivityPattern: Cohen's kappa = 1.000
Currentstatus: Cohen's kappa = 0.000


### 3.3 Normalizing Current Status Labels

The original classifications sometimes include explanatory notes
within categorical labels. For example, "Active (by WoC only;
GitHub repo not found)" and "Active" represent the same activity status.

To ensure consistency between percent agreement and Cohen's kappa,
I normalize these labels before calculating agreement.

When both raters assign the same category to every project,
Cohen's kappa is undefined because there is no variation.

In [23]:
# Recalculate Currentstatus kappa with normalized categories

subset = validation[
    validation["Field"] == "Currentstatus"
].copy()

author = subset["AuthorValue"].fillna("N/A").astype(str).str.strip()
validator = subset["ValidatorValue"].fillna("N/A").astype(str).str.strip()

# Remove explanatory qualifiers from categorical labels
author = author.apply(lambda x: x.split(" (")[0])
validator = validator.apply(lambda x: x.split(" (")[0])

kappa = calculate_kappa(author, validator)

print("Author categories:", author.unique())
print("Validator categories:", validator.unique())
print(
    "Currentstatus Cohen's kappa:",
    "NA (undefined)" if pd.isna(kappa) else round(kappa, 3)
)

Author categories: <StringArray>
['Active']
Length: 1, dtype: str
Validator categories: <StringArray>
['Active']
Length: 1, dtype: str
Currentstatus Cohen's kappa: NA (undefined)


### 3.4 Agreement Statistics Summary

This section summarizes pre- and post-adjudication agreement
across the five classification fields.

All six disagreements were resolved through independent adjudication,
as authorized by the TA. Therefore, post-adjudication agreement
is 100% for all five fields.

Cohen's kappa is reported only for categorical fields.
For Currentstatus, kappa is undefined because all projects
were classified as Active by both raters.

In [24]:
stats = pd.DataFrame([
    {
        "Field": "HasRecovered",
        "PercentAgree_Pre": 70.0,
        "PercentAgree_Post": 100.0,
        "CohenKappa_Pre": 0.286,
        "CohenKappa_Post": "NA",
        "NumDisagreements": 3,
        "NumResolved": 3,
        "Notes": (
            "Qualified Yes labels normalized. "
            "Post-adjudication kappa not calculated because "
            "the final values are independently adjudicated."
        )
    },
    {
        "Field": "WhyRecovered",
        "PercentAgree_Pre": 70.0,
        "PercentAgree_Post": 100.0,
        "CohenKappa_Pre": "NA",
        "CohenKappa_Post": "NA",
        "NumDisagreements": 3,
        "NumResolved": 3,
        "Notes": "Free-text explanations; kappa not applicable."
    },
    {
        "Field": "ActivityPattern",
        "PercentAgree_Pre": 100.0,
        "PercentAgree_Post": 100.0,
        "CohenKappa_Pre": 1.0,
        "CohenKappa_Post": "NA",
        "NumDisagreements": 0,
        "NumResolved": 0,
        "Notes": (
            "All classifications matched. "
            "Post-adjudication kappa not calculated."
        )
    },
    {
        "Field": "Currentstatus",
        "PercentAgree_Pre": 100.0,
        "PercentAgree_Post": 100.0,
        "CohenKappa_Pre": "NA",
        "CohenKappa_Post": "NA",
        "NumDisagreements": 0,
        "NumResolved": 0,
        "Notes": (
            "All projects classified Active; "
            "kappa undefined due to no category variation."
        )
    },
    {
        "Field": "RecentThemes",
        "PercentAgree_Pre": 100.0,
        "PercentAgree_Post": 100.0,
        "CohenKappa_Pre": "NA",
        "CohenKappa_Post": "NA",
        "NumDisagreements": 0,
        "NumResolved": 0,
        "Notes": "Multi-label themes; kappa not applicable."
    }
])

stats.to_csv("aalkafou_agreement_stats.csv", index=False)

print("Created: aalkafou_agreement_stats.csv")
print("Rows:", len(stats))

display(stats)

Created: aalkafou_agreement_stats.csv
Rows: 5


,Field,PercentAgree_Pre,PercentAgree_Post,CohenKappa_Pre,CohenKappa_Post,NumDisagreements,NumResolved,Notes
0,HasRecovered,70.0,100.0,0.286,NA,3,3,Qualified Yes labels normalized. Post-adjudication kappa not calculated because the final values are independently adjudicated.
1,WhyRecovered,70.0,100.0,NA,NA,3,3,Free-text explanations; kappa not applicable.
2,ActivityPattern,100.0,100.0,1.0,NA,0,0,All classifications matched. Post-adjudication kappa not calculated.
3,Currentstatus,100.0,100.0,NA,NA,0,0,All projects classified Active; kappa undefined due to no category variation.
4,RecentThemes,100.0,100.0,NA,NA,0,0,Multi-label themes; kappa not applicable.


## Part 4: Validation Reflection

This section reflects on the validation process, including the
patterns observed across projects, sources of disagreement,
limitations of the available evidence, and lessons learned
about qualitative analysis.

In [25]:
from pathlib import Path

reflection = """# MiniProject3 — Validation Reflection

## Validation Process

For MiniProject3, I reviewed ten projects originally analyzed by my
assigned partner, wknepp. I evaluated five fields for each project:
HasRecovered, WhyRecovered, ActivityPattern, Currentstatus, and
RecentThemes. I compared the original classifications with the
available project statistics, commit histories, and supporting notes.

My partner did not respond to my collaboration email. The TA
authorized me to use my independent judgments as the adjudicated
values where the assignment required consensus. Therefore, the
resolutions in this submission represent independent adjudication,
not an actual discussion with my partner.

## Patterns and Disagreements

The validation produced 50 comparisons, with 44 initial agreements
and six disagreements. All six disagreements concerned recovery:
three involved HasRecovered and three involved WhyRecovered.
They occurred in three projects: olofk_serv, superelastix_elastix,
and anybotics_grid_map.

These projects had relatively short zero-commit gaps of one or two
months, followed by substantial commit activity. My interpretation
classified this as resumed activity, while the original analysis
treated recovery as not applicable. This illustrates how different
interpretations of a gap can affect qualitative classifications.

## Recovery and Activity Patterns

The review showed that a temporary absence of commits does not
necessarily indicate project abandonment. Some projects continued
with hundreds or thousands of commits after their longest recorded
gap. However, the number of subsequent commits alone cannot
establish why activity resumed or whether a meaningful recovery
occurred.

The projects also displayed different activity patterns, including
declining, rising, irregular, and U-shaped development. These
patterns highlight why activity should be evaluated over time rather
than from a single summary statistic.

## Challenges and Limitations

One challenge was distinguishing categorical classifications from
their explanatory qualifiers. For example, "Yes (minimal)" and
"Yes" describe the same basic recovery category, while an
"Active" label with an explanatory note still represents Active.
Normalizing these labels helped avoid misleading disagreement
statistics.

Another challenge was interpreting short inactivity periods.
A one-month gap may simply reflect normal variation in development,
rather than a significant interruption. My recovery decisions used
a literal resumed-activity interpretation, which may differ from
a stricter definition of project recovery.

Some interpretations also relied on the original project's summary
notes and aggregate commit statistics rather than independently
reconstructing every time-series plot. Consequently, agreement
should not be interpreted as proof that every original classification
is objectively correct.

## Agreement Statistics and Lessons Learned

Pre-adjudication agreement was 70% for HasRecovered and
WhyRecovered, and 100% for ActivityPattern, Currentstatus, and
RecentThemes. Cohen's kappa was approximately 0.286 for
HasRecovered and 1.000 for ActivityPattern. For Currentstatus,
kappa was undefined because both sets of classifications contained
only the Active category after normalization.

All six disagreements were independently adjudicated using the
validator's values, resulting in 100% post-adjudication agreement.
This final percentage represents resolution of the recorded
disagreements, not a new independent measurement of reliability.

Overall, this project demonstrated the importance of clear coding
definitions, consistent category normalization, and evidence-based
explanations. It also showed that qualitative validation involves
more than checking whether labels match: the reasoning behind each
classification matters.
"""

Path("aalkafou_validation_reflection.md").write_text(
    reflection,
    encoding="utf-8"
)

print("Created: aalkafou_validation_reflection.md")
print("Word count:", len(reflection.split()))

Created: aalkafou_validation_reflection.md
Word count: 493


## Part 5: Submission Verification

This section verifies that all required MiniProject3 deliverables
were generated successfully and are available in the project directory.

In [26]:
from pathlib import Path

required_files = [
    "aalkafou_validation_log.csv",
    "aalkafou_disagreement_resolution.csv",
    "aalkafou_disagreement.md",
    "aalkafou_agreement_stats.csv",
    "aalkafou_validation_reflection.md"
]

print("MiniProject3 Submission Checklist\n")

for filename in required_files:
    path = Path(filename)
    status = "FOUND" if path.is_file() else "MISSING"
    print(f"{status}: {filename}")

print("\nFiles found:",
      sum(Path(f).is_file() for f in required_files),
      "/", len(required_files))

MiniProject3 Submission Checklist

FOUND: aalkafou_validation_log.csv
FOUND: aalkafou_disagreement_resolution.csv
FOUND: aalkafou_disagreement.md
FOUND: aalkafou_agreement_stats.csv
FOUND: aalkafou_validation_reflection.md

Files found: 5 / 5


### 5.1 Coding Guide Verification

Before submission, I review the coding guide to confirm that my
interpretation of recovery and inactivity gaps follows the
assignment's classification rules.

In [27]:
import pandas as pd

guide = pd.read_csv("CodingGuide.csv")

print("Coding guide columns:")
print(guide.columns.tolist())

display(guide)

Coding guide columns:
['Codebook for conducting validation;;']


,Codebook for conducting validation;;
0,attribute;values;Explanation
1,";;""Read the commit messages carefully looking for any hints about the reasons for the gap and reasons for the interruption of the gap. In addition; check the project’s GitHub issues; pull requests; and README updates immediately before and after the longest gap especially if the reasons are not completely obvious from the commit messages to clarifies why the project went inactive or became active again. For example; issues may show unanswered bug reports; which could be a possible cause of abandonment. A new pull request after the gap might be evidence of recovery. README changes may indicate maintainers marking the project as """"archived"""" or """"revived."""""""
2,"HasRecovered;yes/no;""Did the project become active again after the longest gap? Evidence includes new commits; resumed issue responses; or updated README marking the project as revived."""
3,"WhyRecovered;reason;""If HasRecovered = yes; describe what triggered recovery. Obtain or infer reason from new commits; resumed issue responses; releases or updated README marking the project as revived."""
4,"ActivityPattern;steady/ rising/ declining/ U-shaped/ stopped/ cyclical /irregular;""Categorize the Project’s long term activity trends"
5,Steady: Consistent activity or commit frequency over time; with no significant upward or downward trend. An upward trend might indicate that a person is becoming more active over several months; while a downward trend could signal a decrease in mobility.
6,Rising: Sustained and long-term increase in activity; indicating growing engagement or development momentum.
7,Declining: Gradual long term reduction in activity; suggesting waning participation or project slowdown.
8,U-Shaped: A period of decline followed by recovery; where activity levels drop and later rise again.
9,Stopped: Activity has ceased with no signs of recovery or further engagement.


### 5.2 RecentThemes Agreement Verification

The coding guide includes RecentThemes among the categorical
fields for Cohen's kappa.

I treat each complete theme combination as a category for this
calculation. This measures agreement on the full classification,
rather than agreement on individual themes.

In [28]:
subset = validation[
    validation["Field"] == "RecentThemes"
].copy()

author = subset["AuthorValue"].fillna("N/A").astype(str).str.strip()
validator = subset["ValidatorValue"].fillna("N/A").astype(str).str.strip()

kappa = calculate_kappa(author, validator)

print("RecentThemes agreement:",
      round((author == validator).mean() * 100, 1), "%")

print("RecentThemes Cohen's kappa:",
      "NA (undefined)" if pd.isna(kappa) else round(kappa, 3))

RecentThemes agreement: 100.0 %
RecentThemes Cohen's kappa: 1.0


In [ ]:
### 5.3 Final Agreement Statistics Correction

The coding guide identifies RecentThemes as a categorical field for
Cohen's kappa calculation.

The pre-adjudication Cohen's kappa for RecentThemes is 1.000,
indicating perfect agreement across all ten projects when complete
theme combinations are treated as categories.